In [5]:
import networkx as nx
import random
import math
from itertools import permutations

WIDTH = 70

def line():
    print("-" * WIDTH)

def subline():
    print("-" * WIDTH)

def title(text):
    line()
    print(text.center(WIDTH))
    line()

def section(text):
    print("\n" + text.center(WIDTH))
    line()

#Graph Setup
random.seed(42)

G = nx.Graph()
positions = {}

location_names = {
    0:  "Faizabad",
    1:  "Chandni Chowk",
    2:  "F-10 Markaz",
    3:  "G-9 Markaz",
    4:  "Centaurus",
    5:  "Blue Area",
    6:  "F-6 Supermarket",
    7:  "F-7 Markaz",
    8:  "Pir Wadhai",
    9:  "Saddar Rawalpindi",
    10: "Committee Chowk",
    11: "Peshawar Morr",
    12: "Zero Point",
    13: "Bahria Town",
    14: "DHA Phase 2"
}

num_nodes = 15

for i in range(num_nodes):
    x, y = random.uniform(0, 100), random.uniform(0, 100)
    positions[i] = (x, y)
    G.add_node(i, pos=(x, y))

for i in range(num_nodes):
    for j in range(i + 1, num_nodes):
        x1, y1 = positions[i]
        x2, y2 = positions[j]
        d = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        if d < 75:
            G.add_edge(i, j, weight=round(d, 2))

#Helper Function
def node_distance(a, b):
    x1, y1 = positions[a]
    x2, y2 = positions[b]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def route_text(path):
    return " -> ".join(location_names[n] for n in path)

def heuristic(a, b):
    return node_distance(a, b)

def get_int_input(prompt, lo=0, hi=num_nodes - 1):
    while True:
        try:
            val = int(input(prompt))
            if lo <= val <= hi:
                return val
            print(f"  Please enter a number between {lo} and {hi}.")
        except ValueError:
            print("  Invalid input. Please enter a number.")

def show_locations():
    print(f"\n  {'NODE':<8}{'LOCATION'}")
    subline()
    for i, name in location_names.items():
        print(f"  {i:<8}{name}")
    print()

#Algorithms
#Diikstra Algirithm
def find_driver(driver, pickup):
    path = nx.dijkstra_path(G, driver, pickup, weight="weight")
    dist = nx.dijkstra_path_length(G, driver, pickup, weight="weight")
    return path, dist

#TSP(Travel Salesman Problem) Algorithm
def tsp(driver, pickups):
    unique = list(dict.fromkeys(pickups))
    if len(unique) == 1:
        return unique, nx.dijkstra_path_length(G, driver, unique[0], weight="weight")

    best_order, best_cost = None, float("inf")

    for perm in permutations(unique):
        cost = 0
        cur  = driver
        ok   = True
        for p in perm:
            try:
                cost += nx.dijkstra_path_length(G, cur, p, weight="weight")
                cur = p
            except nx.NetworkXNoPath:
                ok = False
                break
        if ok and cost < best_cost:
            best_cost = cost
            best_order = list(perm)

    if best_order is None:
        best_order = unique

    return best_order, best_cost

#A* Algorithm
def astar(a, b):
    if a == b:
        return [a], 0
    path = nx.astar_path(G, a, b, heuristic=heuristic, weight="weight")
    dist = nx.astar_path_length(G, a, b, heuristic=heuristic, weight="weight")
    return path, dist

#Passenger Class
class Passenger:
    def __init__(self, name, pickup, drop):
        self.name   = name
        self.pickup = pickup
        self.drop   = drop

title("DYNAMIC RIDE SHARING SYSTEM")

section("USER REGISTRATION")
print("  1. Driver")
print("  2. Passenger")

while True:
    user_type = input("\n  Enter your choice (1 or 2): ").strip()
    if user_type in ("1", "2"):
        break
    print("  Invalid choice. Enter 1 or 2.")

#Driver Mode
if user_type == "1":
    section("DRIVER REGISTRATION")
    driver_name    = input("  Driver Name: ").strip()
    driver_vehicle = input("  Vehicle Number: ").strip()
    print(f"\n  Welcome {driver_name}! (Vehicle: {driver_vehicle})")

    show_locations()
    driver_node = get_int_input("  Enter Your Current Location (Node Number): ")

    section("DRIVER MODE — AWAITING PASSENGERS")
    print(f"  {driver_name} is online at {location_names[driver_node]}")

    passengers = []
    n = get_int_input("\n  Enter Number of Passengers to Pick Up: ", lo=1, hi=10)

    for i in range(n):
        section(f"PASSENGER {i + 1}")
        name = input("  Passenger Name: ").strip()
        while not name:
            name = input("  Name cannot be empty. Passenger Name: ").strip()

        show_locations()
        pickup = get_int_input("  Pickup Node: ")
        drop   = get_int_input("  Drop Node:   ")
        while drop == pickup:
            print("  Drop location must be different from pickup.")
            drop = get_int_input("  Drop Node:   ")

        passengers.append(Passenger(name, pickup, drop))
        print(f"\n  {name} registered.")
        print(f"  Pickup : {location_names[pickup]}")
        print(f"  Drop   : {location_names[drop]}")

#Passenger Mode
else:
    section("PASSENGER REGISTRATION")
    passenger_name    = input("  Passenger Name: ").strip()
    passenger_contact = input("  Contact Number: ").strip()
    print(f"\n  Welcome {passenger_name}! (Contact: {passenger_contact})")

    show_locations()
    pickup_node = get_int_input("  Enter Your Pickup Location (Node Number): ")
    drop_node   = get_int_input("  Enter Your Drop Location   (Node Number): ")
    while drop_node == pickup_node:
        print("  Drop location must be different from pickup.")
        drop_node = get_int_input("  Enter Your Drop Location (Node Number): ")

    #Check the nearest driver node using Dijkstra algorithm
    best_driver_node, best_dist = 0, float("inf")
    for candidate in range(num_nodes):
        try:
            d = nx.dijkstra_path_length(G, candidate, pickup_node, weight="weight")
            if d < best_dist:
                best_dist = d
                best_driver_node = candidate
        except nx.NetworkXNoPath:
            pass

    driver_node  = best_driver_node
    driver_name  = "Auto-Assigned Driver"
    passengers   = [Passenger(passenger_name, pickup_node, drop_node)]

    section("PASSENGER MODE — RIDE REQUESTED")
    print(f"  {passenger_name}: {location_names[pickup_node]} -> {location_names[drop_node]}")
    print(f"\n  Nearest driver assigned at : {location_names[driver_node]}")
    print(f"  Distance to your pickup    : {round(best_dist, 2)} units")

#STEP:01 using DIJKSTRA to tell Driver about first pickup
pickup_nodes = [p.pickup for p in passengers]
tsp_order, _ = tsp(driver_node, pickup_nodes)
first_pickup  = tsp_order[0]

dpath, ddist = find_driver(driver_node, first_pickup)

section("DIJKSTRA ALGORITHM — DRIVER TO FIRST PICKUP")
print(f"  Driver Location : {location_names[driver_node]}")
print(f"  First Pickup    : {location_names[first_pickup]}")
print(f"  Distance        : {round(ddist, 2)} units")
print(f"  Route           : {route_text(dpath)}")

#STEP:02 using TS to tell the Optimized Pickup Order
section("TSP ALGORITHM — OPTIMIZED PICKUP ORDER")
print("  Optimal order to collect all passengers:\n")
for idx, node in enumerate(tsp_order, 1):
    names = [p.name for p in passengers if p.pickup == node]
    label = ", ".join(names) if names else ""
    print(f"  {idx}. {location_names[node]}  ({label})")

# STEP: 03 using A* to find the Full Route
section("A* ALGORITHM — COMPLETE ROUTE CALCULATION")

current    = driver_node
total      = 0
full_route = [driver_node]

#Reorder passengers to match TSP pickup order
ordered_passengers = []
for node in tsp_order:
    for p in passengers:
        if p.pickup == node and p not in ordered_passengers:
            ordered_passengers.append(p)

#Pickup segments
print("\n  PICKUP ROUTES")
subline()

for node in tsp_order:
    path, dist = astar(current, node)
    names = [p.name for p in passengers if p.pickup == node]
    label = ", ".join(names)

    if dist > 0:
        print(f"\n  Picking up : {label}")
        print(f"  From       : {location_names[current]}")
        print(f"  To         : {location_names[node]}")
        print(f"  Route      : {route_text(path)}")
        print(f"  Distance   : {round(dist, 2)} units")
        full_route.extend(path[1:])   #used to avoid duplicate
        total += dist
        current = node

    else:
        print(f"\n  {label}: already at {location_names[node]}")

#Drop segments using FIFO order
print("\n  DROP ROUTES")
subline()

for p in ordered_passengers:
    path, dist = astar(current, p.drop)

    if dist > 0:
        print(f"\n  Dropping   : {p.name}")
        print(f"  From       : {location_names[current]}")
        print(f"  To         : {location_names[p.drop]}")
        print(f"  Route      : {route_text(path)}")
        print(f"  Distance   : {round(dist, 2)} units")
        full_route.extend(path[1:])
        total += dist
        current = p.drop
    else:
        print(f"\n  {p.name}: already at drop location {location_names[p.drop]}")

#Final Output
title("FINAL OPTIMIZED SHARED RIDE")
print("  Complete Route:")
print(f"  {route_text(full_route)}")
print(f"\n  Total Distance : {round(total, 2)} units")

#Fare Distribution
BASE = 10
fare = total * BASE

section("FARE DISTRIBUTION")
print(f"  Base Rate  : Rs. {BASE} / unit")
print(f"  Total Fare : Rs. {round(fare, 2)}")
subline()

total_dist_sum = sum(node_distance(p.pickup, p.drop) for p in passengers)

for p in passengers:
    pd    = node_distance(p.pickup, p.drop)
    share = (pd / total_dist_sum) * fare if total_dist_sum > 0 else fare / len(passengers)
    print(f"  {p.name:<20} Rs. {round(share, 2)}")

line()
print("\n  THANK YOU FOR USING OUR SYSTEM".center(WIDTH))
line()

----------------------------------------------------------------------
                     DYNAMIC RIDE SHARING SYSTEM                      
----------------------------------------------------------------------

                          USER REGISTRATION                           
----------------------------------------------------------------------
  1. Driver
  2. Passenger

  Enter your choice (1 or 2): 1

                         DRIVER REGISTRATION                          
----------------------------------------------------------------------
  Driver Name: Khudema
  Vehicle Number: Sonata

  Welcome Khudema! (Vehicle: Sonata)

  NODE    LOCATION
----------------------------------------------------------------------
  0       Faizabad
  1       Chandni Chowk
  2       F-10 Markaz
  3       G-9 Markaz
  4       Centaurus
  5       Blue Area
  6       F-6 Supermarket
  7       F-7 Markaz
  8       Pir Wadhai
  9       Saddar Rawalpindi
  10      Committee Chowk
  11      Peshaw